# Setup

I used this file to train the model in Colab for GPU access

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("No GPU found — go to Runtime > Change runtime type > T4 GPU")

PyTorch version: 2.11.0+cu128
CUDA available: True
Using GPU: Tesla T4


# Imports and install

In [ ]:
!pip install -q kaggle

import os
import numpy as np
import matplotlib.pyplot as plt

import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim

from sklearn.utils.class_weight import compute_class_weight


# Download dataset

In [ ]:
import json
import os

kaggle_creds = {
    "username": "rafadua",
    "key": "KGAT_2421296ffd9cca0c2be023525b8a8b1"
}

kaggle_path = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_path, exist_ok=True)
with open(os.path.join(kaggle_path, "kaggle.json"), "w") as f:
    json.dump(kaggle_creds, f)

!chmod 600 ~/.kaggle/kaggle.json
os.environ['KAGGLE_CONFIG_DIR'] = kaggle_path

In [ ]:
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip -d fer2013_data

Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
100% 60.3M/60.3M [00:00<00:00, 175MB/s]



In [ ]:
!ls fer2013_data
!ls fer2013_data/train

test  train
angry  disgust	fear  happy  neutral  sad  surprise


# Data loading

In [ ]:
# Define the directory paths
train_dir = 'fer2013_data/train'
test_dir = 'fer2013_data/test'

# Training transforms: augmentation helps prevent overfitting
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Ensure 1 channel (grayscale)
    transforms.RandomHorizontalFlip(),            # Data augmentation (randomly flips image horizontally)
    transforms.RandomRotation(10),                # Data augmentation (randomly rotates pixels by 10 degrees)
    transforms.ToTensor(),                        # Convert image to [0, 1] tensor
    transforms.Normalize((0.5,), (0.5,))          # Normalize to [-1, 1]
])

# Validation/Test transforms: No augmentation, just preprocessing
test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load datasets using ImageFolder (infers classes from folder names)
full_train_dataset = ImageFolder(root=train_dir, transform=train_transform)
test_dataset = ImageFolder(root=test_dir, transform=test_transform)

# Split train into Train (90%) and Validation (10%)
train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Classes found: {full_train_dataset.classes}")
print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Testing images: {len(test_dataset)}")

Classes found: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Training images: 25838
Validation images: 2871
Testing images: 7178


# Model Definition

In [ ]:
class EmotionCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(EmotionCNN, self).__init__()

        # Block 1: Input 48x48 -> 24x24
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25)
        )

        # Block 2: 24x24 -> 12x12
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25)
        )

        # Block 3: 12x12 -> 6x6
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25)
        )

        # Fully Connected Layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.classifier(x)
        return x

# Initialize and display parameter count
model = EmotionCNN(num_classes=7).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

Total Parameters: 2,502,887


# Class weights

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_labels = [label for _, label in train_dataset]
class_names = full_train_dataset.classes

unique_labels = np.unique(train_labels)
weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_labels,
    y=train_labels
)

class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

print("Class weights calculated and loss function initialized successfully.")
print(f"Weights: {dict(zip(class_names, weights.round(4)))}")

Class weights calculated and loss function initialized successfully.
Weights: {'angry': np.float64(1.0256), 'disgust': np.float64(9.3684), 'fear': np.float64(1.0063), 'happy': np.float64(0.5679), 'neutral': np.float64(0.8259), 'sad': np.float64(0.8429), 'surprise': np.float64(1.3048)}


# Training

In [ ]:
num_epochs = 100
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train()
    train_loss, train_correct = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels.data)

    epoch_train_loss = train_loss / len(train_dataset)
    epoch_train_acc = train_correct.double() / len(train_dataset)

    # --- Validation Phase ---
    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data)

    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_correct.double() / len(val_dataset)

    # Update Scheduler
    scheduler.step(epoch_val_loss)

    # Save history
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc.item())
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc.item())

    print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}')

    # Save Best Model
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), 'best.pt')
        print("==> Best model saved")

# save to google drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('best.pt', '/content/drive/MyDrive/best.pt')

# --- Plotting Results ---
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend(); plt.title('Loss')

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.legend(); plt.title('Accuracy')
plt.show()

Epoch 1/100 | Train Loss: 1.4337 Acc: 0.4275 | Val Loss: 1.3069 Acc: 0.4908
==> Best model saved
Epoch 2/100 | Train Loss: 1.4303 Acc: 0.4288 | Val Loss: 1.3058 Acc: 0.5305
==> Best model saved
Epoch 3/100 | Train Loss: 1.4032 Acc: 0.4420 | Val Loss: 1.3253 Acc: 0.5127
Epoch 4/100 | Train Loss: 1.4224 Acc: 0.4369 | Val Loss: 1.2813 Acc: 0.5002
Epoch 5/100 | Train Loss: 1.3961 Acc: 0.4436 | Val Loss: 1.3057 Acc: 0.5211
Epoch 6/100 | Train Loss: 1.3920 Acc: 0.4429 | Val Loss: 1.2907 Acc: 0.5051
Epoch 7/100 | Train Loss: 1.3955 Acc: 0.4411 | Val Loss: 1.3240 Acc: 0.4984
Epoch 8/100 | Train Loss: 1.3701 Acc: 0.4484 | Val Loss: 1.2903 Acc: 0.5193
Epoch 9/100 | Train Loss: 1.3790 Acc: 0.4482 | Val Loss: 1.2646 Acc: 0.5232
Epoch 10/100 | Train Loss: 1.3569 Acc: 0.4565 | Val Loss: 1.2646 Acc: 0.5152
Epoch 11/100 | Train Loss: 1.3501 Acc: 0.4635 | Val Loss: 1.2659 Acc: 0.5172
Epoch 12/100 | Train Loss: 1.3206 Acc: 0.4717 | Val Loss: 1.2660 Acc: 0.5277
Epoch 13/100 | Train Loss: 1.3132 Acc: 0.47

In [ ]:
# 5. FINAL EVALUATION
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Load the best saved weights
best_model = EmotionCNN(num_classes=7).to(device)
best_model.load_state_dict(torch.load('best.pt'))
best_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = best_model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Metrics
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()